# 🎙️ AI Voiceover Studio su Colab — Pixabay + overlay grafici

Webapp completa (script → voce → video, tab **AI Video** e **Video wizard**) che gira su Google Colab.
Questa versione è impostata per lavorare **senza YouTube**: le clip arrivano da **Pixabay**
(video stock professionali, taggati) e lo stile presentazione usa **card trasparenti disegnate
localmente** (testi, icone e linee — nessun modello di immagini richiesto).

**Come si usa:**
1. Aggiungi due secret (icona 🔑 a sinistra → *Secrets*):
   - `OPENROUTER_API_KEY` — per la narrazione, la selezione dei clip e lo script;
   - `PIXABAY_KEY` — chiave gratuita, visibile in cima a [pixabay.com/api/docs/](https://pixabay.com/api/docs/) quando sei registrato.
2. Esegui le celle in ordine e apri il link ***.gradio.live* che compare alla fine.


In [ ]:
# 1) Installazione (~1 minuto)
%%bash
set -e
ffmpeg -version | head -1
pip install -q "ai-voiceover-studio[webui,edge] @ git+https://github.com/mnatali74/ai-voiceover-studio.git@vibe/web-studio-b186e9"
echo "Installazione completata."


In [ ]:
# 2) Chiavi API (dai Secrets di Colab, icona 🔑 a sinistra)
import os

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("OPENROUTER_API_KEY: ok")
except Exception:
    if not os.environ.get("OPENROUTER_API_KEY"):
        print("⚠️ Secret OPENROUTER_API_KEY mancante: senza chiave non funzionano la")
        print("   selezione dei clip, lo script e il generatore di illustrazioni.")

try:
    from google.colab import userdata
    os.environ["VOICEOVER_PIXABAY_KEY"] = userdata.get("PIXABAY_KEY")
    print("PIXABAY_KEY: ok")
except Exception:
    if not os.environ.get("VOICEOVER_PIXABAY_KEY"):
        print("⚠️ Secret PIXABAY_KEY mancante: la fonte Pixabay non funzionerà.")
        print("   Chiave gratuita, visibile in cima a pixabay.com/api/docs/ da loggato.")


In [ ]:
# 3) Verifica delle chiavi e diagnosi (consigliata: riproduce il bottone 'Suggest')
from voiceover_studio.ai_video import OpenRouterClient, best_value_models
from voiceover_studio.clips import search_pixabay, suggest_searches

try:
    client = OpenRouterClient()
    print("OpenRouter: ok,", len(client.models()), "modelli disponibili")
    model = best_value_models(client)[0]
    print("Modello 'best value':", model)
    queries = suggest_searches(client, model, "enoturismo nelle Marche", n=3)
    print("Suggest queries: ok ->", queries)
except Exception as exc:
    print("OpenRouter/Suggest NON funziona:", type(exc).__name__)
    print(str(exc)[:500])

try:
    found = search_pixabay("nature", "")  # chiave vuota -> usa il Secret PIXABAY_KEY
    print("Pixabay: ok,", len(found), "video trovati per una ricerca di prova")
except Exception as exc:
    print("Pixabay NON funziona:", type(exc).__name__, str(exc)[:300])


### Fonti e stile presentazione (senza YouTube)

- **Pixabay** (consigliata): video stock cinematografici con descrizioni precise, nessun bot-check.
  Nel tab *Video wizard* spunta solo **Pixabay** in *Sources to search* (la chiave letta dai
  Secrets viene presa automaticamente; puoi anche incollarla nel campo della UI).
- **Wikimedia Commons** (opzionale): keyless, video e immagini con descrizioni ricche.
- **Overlay in stile presentazione**: dopo "6. Select the best clips and write the script",
  premi *7. Add overlays to the script*: le card trasparenti (titolo, punti chiave, icone,
  linee di accento) vengono **disegnate in locale** e si sovrappongono al video, che continua
  a girare sotto. Nessun modello di immagini e nessun costo extra.

- **AI polish (stile HeyGen)**: dopo l'anteprima, premi *9. AI polish* — il modello guarda i
  frame del video, assegna un punteggio di professionalità e corregge ciò che può (colori,
  card illeggibili ridisegnate, clip sostituite con una più adatta), poi rirenderizza e
  ricontrolla finché il risultato non è buono.


In [ ]:
# 4) Avvia lo studio — apri il link *.gradio.live che compare qui sotto
# Se la porta è occupata da un avvio precedente, ne viene scelta automaticamente una libera.
import socket
from pathlib import Path

from voiceover_studio.webui import launch


def free_port(start=7860, tries=20):
    for port in range(start, start + tries):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    return start

launch(workdir=Path("studio_out"), share=True, port=free_port())  # interrompi la cella per fermare il server


In [ ]:
# 5) (Opzionale) Scarica i risultati come zip
import shutil
from pathlib import Path

out = Path("studio_out")
videos = sorted(out.rglob("*.mp4")) + sorted(out.rglob("*.wav"))
print("\n".join(str(v) for v in videos[:40]) or "Nessun file ancora.")
if any(out.iterdir()):
    shutil.make_archive("studio_results", "zip", out)
    print("Creato studio_results.zip: scaricalo dal pannello File a sinistra.")
